# Bayesian Networks

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/graphical-models/01-bayesian-networks

A from-scratch, runnable implementation of the concepts in the lesson.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## Intuition — a joint distribution, factored by structure

A joint distribution over `n` binary variables needs `2ⁿ − 1` numbers — hopeless at scale. A
**Bayesian network** exploits *conditional independence*: draw the variables as a DAG where each node
depends only on its **parents**, and the joint factorizes into small conditional tables:
`P(C,S,R,W) = P(C)·P(S|C)·P(R|C)·P(W|S,R)`. The classic **sprinkler network** (Cloudy → Sprinkler,
Rain → WetGrass) shows everything: exact inference by enumeration, the strange-but-correct
**explaining away** effect, and the exponential parameter savings from structure. We build it all from
scratch and verify the inferences by Monte Carlo simulation.

## The Sprinkler network — exact inference from scratch

Variables: Cloudy $C$, Sprinkler $S$, Rain $R$, Wet grass $W$, with $P(C,S,R,W)=P(C)P(S|C)P(R|C)P(W|S,R)$. We enumerate the full joint and answer queries by brute force — no library.

In [ ]:
P_C = [0.5, 0.5]                       # P(C=0), P(C=1)
P_S = [[0.5, 0.5], [0.9, 0.1]]          # P(S=s | C=c)
P_R = [[0.8, 0.2], [0.2, 0.8]]          # P(R=r | C=c)
# P(W=w | S=s, R=r)
P_W = [[[1.0, 0.0], [0.1, 0.9]], [[0.1, 0.9], [0.01, 0.99]]]

def joint(c, s, r, w):
    return P_C[c] * P_S[c][s] * P_R[c][r] * P_W[s][r][w]

# sanity: full joint sums to 1
total = sum(joint(c,s,r,w) for c in (0,1) for s in (0,1) for r in (0,1) for w in (0,1))
print('joint sums to', round(total, 6))

**What to notice:** the entire network is four small tables — a prior for Cloudy and one conditional
table per node given its parents — and their product defines the full joint (which correctly sums
to 1). The DAG structure *is* the claim that, e.g., Sprinkler and Rain are independent **given**
Cloudy; the factorization encodes it.

## Querying by enumeration

$P(R=1\mid W=1)=\dfrac{\sum_{c,s}P(c,s,R{=}1,W{=}1)}{\sum_{c,s,r}P(c,s,r,W{=}1)}$ — sum out the unobserved variables, then normalize.

In [ ]:
def query(evidence, target):
    """P(target_var = target_val | evidence dict)."""
    tv, tval = target
    num = den = 0.0
    for c in (0,1):
        for s in (0,1):
            for r in (0,1):
                for w in (0,1):
                    assign = dict(C=c, S=s, R=r, W=w)
                    if any(assign[k] != v for k, v in evidence.items()):
                        continue
                    p = joint(c, s, r, w)
                    den += p
                    if assign[tv] == tval: num += p
    return num / den

print('P(Rain=1)            =', round(query({}, ('R',1)), 3))
print('P(Rain=1 | Wet=1)    =', round(query({'W':1}, ('R',1)), 3), ' (wet grass raises rain)')

**What to notice:** inference by **enumeration** is conceptually trivial — sum the joint over all
assignments consistent with the evidence, then normalize. Observing wet grass raises `P(Rain)` from
0.5 to ~0.7: evidence flows *backward* through the network, from effect to cause, exactly as Bayes'
rule dictates.

## The library way — verify by forward-sampling Monte Carlo

Exact enumeration should agree with brute-force simulation: sample the network **forward** (parents
before children) millions of times, then just *count* the conditional frequencies. The cell checks the
two match — the standard way to validate any inference engine.

In [ ]:
rng = np.random.default_rng(0)
N = 400_000
C = rng.random(N) < 0.5
S = np.where(C, rng.random(N) < 0.1, rng.random(N) < 0.5)     # P(S=1|C=1)=0.1, P(S=1|C=0)=0.5
Rn = np.where(C, rng.random(N) < 0.8, rng.random(N) < 0.2)    # P(R=1|C)
pw = np.select([S & Rn, S & ~Rn, ~S & Rn], [0.99, 0.9, 0.9], default=0.0)
W = rng.random(N) < pw

mc_rain_wet = Rn[W].mean()
mc_rain_wet_spr = Rn[W & S].mean()
print(f'P(R=1|W=1)      exact {query({"W":1}, ("R",1)):.3f}   Monte Carlo {mc_rain_wet:.3f}')
print(f'P(R=1|W=1,S=1)  exact {query({"W":1,"S":1}, ("R",1)):.3f}   Monte Carlo {mc_rain_wet_spr:.3f}')
assert abs(mc_rain_wet - query({'W':1}, ('R',1))) < 0.01
assert abs(mc_rain_wet_spr - query({'W':1,'S':1}, ('R',1))) < 0.01
print('\nexact enumeration == forward-sampling Monte Carlo ✓')

**What to notice:** counting frequencies in 400k simulated worlds reproduces the exact enumeration to
two decimals — two completely independent methods, one answer. (Real libraries like `pgmpy` do the
enumeration/variable-elimination for you; the simulation check works on *any* engine.)

## Explaining away

Rain and Sprinkler are marginally independent causes of wet grass. Once we observe wet grass, learning the sprinkler was on should **lower** the probability of rain.

In [ ]:
p_rain_wet      = query({'W':1}, ('R',1))
p_rain_wet_spr  = query({'W':1, 'S':1}, ('R',1))
print(f'P(Rain | Wet)            = {p_rain_wet:.3f}')
print(f'P(Rain | Wet, Sprinkler) = {p_rain_wet_spr:.3f}')
print('-> sprinkler explains away the wet grass, so rain becomes less likely.')

**What to notice:** **explaining away** — learning the sprinkler was on *drops* `P(Rain|Wet)` from
~0.7 to ~0.32. Rain and Sprinkler are independent a priori, but become **dependent given their common
effect** (wet grass): one cause being confirmed makes the other less necessary. This
collider/v-structure behavior is the signature (and least intuitive) inference pattern of Bayesian
networks.

## Parameter savings from factorization

In [ ]:
full = 2**4 - 1
factored = 1 + 2 + 2 + 4    # P(C) + P(S|C) + P(R|C) + P(W|S,R)
print(f'full joint table: {full} free parameters | factorized: {factored}')

**What to notice:** the factored network needs **9** parameters where the full joint needs **15** —
and the gap explodes with size: 20 binary variables need ~10⁶ joint parameters but only tens in a
sparse network. Structure is what makes probabilistic modeling scale.

## Gotchas & tradeoffs

- **Enumeration is exponential.** Summing over all `2ⁿ` assignments works for 4 variables, not 40 —
  real inference uses variable elimination, belief propagation, or sampling.
- **Structure encodes independence claims** — wrong structure means wrong inferences, no matter how
  good the tables. (d-separation, in the lesson's viz, reads independences off the graph.)
- **Colliders invert intuition:** conditioning on a common *effect* creates dependence (explaining
  away); conditioning on a common *cause* removes it. Easy to get backwards.
- **Edges are not causality** by themselves — a DAG encodes conditional independence; causal reading
  requires extra assumptions (the causal-inference course).

In [ ]:
# Enumeration cost doubles per variable: fine at n=4, hopeless at n=40
for n in [4, 10, 20, 40]:
    print(f'n={n:>2} binary variables -> {2**n:>15,} joint assignments to enumerate')
print('\n-> exact enumeration is exponential; scalable inference needs elimination/BP/sampling')

**What to notice:** at 40 variables enumeration means a trillion terms — the from-scratch method here
is for understanding, not scale. Variable elimination and belief propagation (lesson 3's HMM machinery
is a special case) exploit the graph structure to avoid the blow-up.

## Key takeaways

- A Bayesian network factorizes the joint into per-node conditionals over a DAG.
- Exact inference by enumeration: sum out unobserved variables, then normalize.
- Observing a common effect couples its causes — **explaining away**.
- Factorization turns an exponential table into a few small CPTs.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Counting parameters

A CPT for a $k$-state node needs $(k-1)$ free numbers **per combination of parent values**:

$$\#\text{params} = (k - 1) \prod_{p \in \text{parents}} |p|$$

Implement that, plus the unfactorized joint's count ($\prod |X_i| - 1$). The checks redo the sprinkler arithmetic (9 vs 15) and then scale it up: a 20-node binary chain needs **39** numbers where the raw joint needs over a million.

In [ ]:
def cpt_params(k, parent_cards):
    """Free parameters of a CPT: node with k states, parents with the given cardinalities."""
    # TODO(you): (k - 1) times the product of parent cardinalities
    n = 1
    for c in parent_cards:
        n *= c
    return ...


def full_joint_params(cards):
    """Free parameters of the unfactorized joint over variables with these cardinalities."""
    # TODO(you): product of all cardinalities, minus 1
    return ...

In [ ]:
# Checks — run me
sprinkler = cpt_params(2, []) + cpt_params(2, [2]) + cpt_params(2, [2]) + cpt_params(2, [2, 2])
assert sprinkler == 9, "C(1) + S|C(2) + R|C(2) + W|S,R(4) = 9 free parameters"
assert full_joint_params([2, 2, 2, 2]) == 15, "the unfactorized joint needs 15"

assert full_joint_params([2] * 20) == 2 ** 20 - 1, "20 binary variables: over a million"
chain20 = cpt_params(2, []) + 19 * cpt_params(2, [2])
assert chain20 == 39, "a 20-node chain: 1 + 19*2 — factorization is exponential savings"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def cpt_params(k, parent_cards):
    n = 1
    for c in parent_cards:
        n *= c
    return (k - 1) * n


def full_joint_params(cards):
    n = 1
    for c in cards:
        n *= c
    return n - 1
```

</details>

### Exercise 2 — The factorized joint

Build the sprinkler network's joint as the product the graph dictates:

$$P(C, S, R, W) = P(C) \, P(S \mid C) \, P(R \mid C) \, P(W \mid S, R)$$

The checks confirm it's a valid distribution (all 16 configurations sum to 1), match one configuration computed by hand, and verify the hard zero: no sprinkler and no rain means the grass cannot be wet.

In [ ]:
P_C = {True: 0.5, False: 0.5}
P_S = {True: 0.1, False: 0.5}        # P(S=T | C)
P_R = {True: 0.8, False: 0.2}        # P(R=T | C)
P_W = {(True, True): 0.99, (True, False): 0.9, (False, True): 0.9, (False, False): 0.0}


def joint(c, s, r, w):
    """P(C=c, S=s, R=r, W=w) from the four factors above."""
    # TODO(you): start with P(C)
    p = ...

    # TODO(you): multiply by P(S=s | C) — use P_S[c] if s else 1 - P_S[c]
    p *= ...

    # TODO(you): multiply by P(R=r | C)
    p *= ...

    # TODO(you): multiply by P(W=w | S, R)
    pw = P_W[(s, r)]
    p *= ...

    return p

In [ ]:
# Checks — run me
total = sum(joint(c, s, r, w) for c in [True, False] for s in [True, False]
            for r in [True, False] for w in [True, False])
assert abs(total - 1) < 1e-12, "the factorized joint sums to 1 over all 16 configurations"

expected = 0.5 * 0.1 * 0.8 * 0.99
assert abs(joint(True, True, True, True) - expected) < 1e-12, "P(C)P(S|C)P(R|C)P(W|S,R) by hand"
assert joint(False, False, False, True) == 0.0, "no sprinkler, no rain -> grass can't be wet"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def joint(c, s, r, w):
    p = P_C[c]
    p *= P_S[c] if s else 1 - P_S[c]
    p *= P_R[c] if r else 1 - P_R[c]
    pw = P_W[(s, r)]
    p *= pw if w else 1 - pw
    return p
```

</details>